In [1]:
from audio_processing import AudioSignal, TimeFeatures, STFTFeatures
import numpy as np
import pandas as pd
import soundfile as sf
import os
import sys

In [2]:
SR = 22050
DURATION = 3.0
N = 2048
H = 512
OUT_DIR = "test_signals"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_sine(freq=440.0, amp=0.5, duration=DURATION, sr=SR):
    """Pure sine - known RMS = amp / sqrt(2)"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_am_sine(carrier_freq=440.0, mod_freq=2.0, mod_depth=0.8, amp=0.5, duration=DURATION, sr=SR):
    """AM modulated sine - envelope varies sinusoidally at mod_freq Hz"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    envelope = amp * (1.0 + mod_depth * np.sin(2 * np.pi * mod_freq * t))
    carrier = np.sin(2 * np.pi * carrier_freq * t)
    return envelope * carrier

def gen_white_noise(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """Gaussian white noise - crest factor typically 4-5x"""
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

def gen_click_train(rate_hz=2.0, amp=0.9, duration=DURATION, sr=SR):
    """Periodic impulses - very high crest, sparse energy"""
    sig = np.zeros(int(sr * duration))
    period_samples = int(sr / rate_hz)
    for i in range(0, len(sig), period_samples):
        decay_len = min(64, len(sig) - i)
        decay = amp * np.exp(-np.linspace(0, 5, decay_len))
        sig[i:i+decay_len] = decay
    return sig

def gen_stepped_energy(n_steps=6, duration=DURATION, sr=SR):
    """Staircase amplitude - DR = 20*log10(max_level/min_level)"""
    n_samples = int(sr * duration)
    step_len = n_samples // n_steps
    levels = [0.05, 0.1, 0.2, 0.4, 0.8, 0.5]
    sig = np.zeros(n_samples)
    t = np.linspace(0, duration, n_samples, endpoint=False)
    sine = np.sin(2 * np.pi * 440.0 * t)
    for i, lvl in enumerate(levels):
        start = i * step_len
        end = start + step_len if i < n_steps - 1 else n_samples
        sig[start:end] = lvl * sine[start:end]
    return sig

def gen_silence_padded(inner_amp=0.5, silence_ratio_target=0.4, duration=DURATION, sr=SR):
    """Sine with silence blocks - known silence ratio"""
    n_samples = int(sr * duration)
    n_silence = int(n_samples * silence_ratio_target)
    n_active = n_samples - n_silence
    t = np.linspace(0, duration, n_samples, endpoint=False)
    sig = inner_amp * np.sin(2 * np.pi * 440.0 * t)
    block_size = n_silence // 3
    positions = [
        (0, block_size),
        (n_active // 2, n_active // 2 + block_size),
        (n_samples - block_size, n_samples)
    ]
    for start, end in positions:
        sig[start:end] = 0.0
    return sig

def gen_percussive_train(onset_rate=4.0, attack_ms=2.0, decay_ms=50.0, amp=0.7, duration=DURATION, sr=SR):
    """Periodic percussive hits - known attack/decay envelope"""
    n_samples = int(sr * duration)
    sig = np.zeros(n_samples)
    period_samples = int(sr / onset_rate)
    attack_samples = int(sr * attack_ms / 1000.0)
    decay_samples = int(sr * decay_ms / 1000.0)
    hit_len = attack_samples + decay_samples
    for i in range(0, n_samples, period_samples):
        end = min(i + hit_len, n_samples)
        actual_len = end - i
        atk_end = min(attack_samples, actual_len)
        attack_env = np.linspace(0, 1, atk_end)
        dec_len = actual_len - atk_end
        decay_env = np.exp(-np.linspace(0, 5, dec_len)) if dec_len > 0 else np.array([])
        envelope = np.concatenate([attack_env, decay_env])
        rng = np.random.default_rng(seed=i)
        burst = rng.standard_normal(actual_len)
        sig[i:end] = amp * envelope * burst
    return sig

In [4]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    Format: { feature_name: (min_expected, max_expected) }
    """
    gt = {}

    sine_rms = 0.5 / np.sqrt(2)
    gt["sine"] = {
        "rms_median":       (sine_rms * 0.9, sine_rms * 1.1),
        "crest_median":     (1.3, 1.6),
        "dynamic_range":    (0.0, 3.0),
        "energy_variance":  (0.0, 0.15),
        "energy_mod_rate":  (0.0, 0.01),
    }

    gt["am_sine"] = {
        "rms_median":       (0.1, 0.6),
        "crest_median":     (1.3, 2.0),
        "dynamic_range":    (5.0, 20.0),
        "energy_variance":  (0.3, 2.0),
        "energy_mod_rate":  (0.001, 0.01),
    }

    gt["white_noise"] = {
        "rms_median":       (0.45, 0.55),
        "crest_median":     (1.8, 2.5),
        "dynamic_range":    (0.0, 1.0),
        "energy_variance":  (0.0, 0.3),
        "energy_mod_rate":  (0.0, 0.1),
    }

    gt["click_train"] = {
        "rms_median":       (0.0, 0.05),
        "crest_median":     (10.0, 200.0),
        "dynamic_range":    (20.0, 60.0),
        "energy_variance":  (0.0, 0.01),
        "energy_mod_rate":  (0.0, 0.01),
    }

    gt["stepped_energy"] = {
        "rms_median":       (0.1, 0.4),
        "crest_median":     (1.3, 1.6),
        "dynamic_range":    (18.0, 28.0),
        "energy_variance":  (0.5, 3.0),
        "energy_mod_rate":  (0.001, 0.01),
    }

    gt["silence_padded"] = {
        "rms_median":       (0.0, 0.4),
        "crest_median":     (1.3, 50.0),
        "dynamic_range":    (60.0, 80.0),
        "energy_variance":  (0.0, 0.1),
        "energy_mod_rate":  (0.0, 0.02),
    }

    gt["percussive"] = {
        "rms_median":       (0.0, 0.15),
        "crest_median":     (3.0, 30.0),
        "dynamic_range":    (60.0, 80.0),
        "energy_variance":  (0.5, 5.0),
        "energy_mod_rate":  (0.05, 0.8),
        "attack_time":      (0.0, 0.005),
    }

    return gt

In [5]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals as .wav files."""
    signals = {
        "sine":             gen_sine(),
        "am_sine":          gen_am_sine(),
        "white_noise":      gen_white_noise(),
        "click_train":      gen_click_train(),
        "stepped_energy":   gen_stepped_energy(),
        "silence_padded":   gen_silence_padded(),
        "percussive":       gen_percussive_train(),
    }
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    return paths

In [6]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Instantiate AudioSignal + TimeFeatures for each file.
    Pulls scalar results into a flat DataFrame.
    
    ADAPT: uncomment the import at the top of this file
    and point sys.path to your project directory.
    """
    # --- UNCOMMENT THESE when you have your module on the path ---
    from audio_processing import AudioSignal, TimeFeatures
    
    results = []
    for name, path in file_paths.items():
        print(f"  Processing: {name} ({path})")

        try:
            # --- 1. Load via AudioSignal ---
            sig = AudioSignal(path, N=N, H=H)

            # --- 2. Check validity BEFORE calling features ---
            if sig.invalid:
                print(f"    [SKIP] AudioSignal marked invalid: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    'rms_median': np.nan,
                    'rms_std': np.nan,
                    'crest_median': np.nan,
                    'crest_std': np.nan,
                    'dynamic_range': np.nan,
                    'energy_variance': np.nan,
                    'energy_mod_rate': np.nan,
                    'attack_time': np.nan,
                    'attack_slope': np.nan,
                    'decay_slope': np.nan,
                })
                continue

            # --- 3. Instantiate TimeFeatures ---
            tf = TimeFeatures(sig)

            # --- 4. Extract arrays, aggregate here (not inside the class) ---
            rms = tf._rms_envelope()
            crest = tf._crest_factor()
            mask = tf._active_rms_mask()

            results.append({
                'signal':            name,
                'invalid':           False,
                'rms_median':        float(np.median(rms)),              # Full envelope - silence IS meaningful here
                'rms_std':           float(np.std(rms)),
                'crest_median':      float(np.median(crest[mask])) if mask.any() else 0.0,  # Active only
                'crest_std':         float(np.std(crest[mask])) if mask.any() else 0.0,     # Active only
                'dynamic_range':     float(tf._dynamic_range()),
                'energy_variance':   float(tf.energy_variance()),
                'energy_mod_rate':   float(tf._energy_modulation_rate()),
                'attack_time':       float(tf._attack_time()),
                'attack_slope':      float(tf._attack_slope()),
                'decay_slope':       float(tf._decay_slope()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                'rms_median': np.nan,
                'rms_std': np.nan,
                'crest_median': np.nan,
                'crest_std': np.nan,
                'dynamic_range': np.nan,
                'energy_variance': np.nan,
                'energy_mod_rate': np.nan,
                'attack_time': np.nan,
                'attack_slope': np.nan,
                'decay_slope': np.nan,
            })

    return pd.DataFrame(results)

In [7]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  AMPLITUDE/ENERGY FEATURE TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.4f}, {r['expected_max']:.4f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [8]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved test_signals\sine.wav | 66150 samples | 3.00s
  Saved test_signals\am_sine.wav | 66150 samples | 3.00s
  Saved test_signals\white_noise.wav | 66150 samples | 3.00s
  Saved test_signals\click_train.wav | 66150 samples | 3.00s
  Saved test_signals\stepped_energy.wav | 66150 samples | 3.00s
  Saved test_signals\silence_padded.wav | 66150 samples | 3.00s
  Saved test_signals\percussive.wav | 66150 samples | 3.00s


In [9]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: sine (test_signals\sine.wav)
  Processing: am_sine (test_signals\am_sine.wav)
  Processing: white_noise (test_signals\white_noise.wav)
  Processing: click_train (test_signals\click_train.wav)
  Processing: stepped_energy (test_signals\stepped_energy.wav)
  Processing: silence_padded (test_signals\silence_padded.wav)
  Processing: percussive (test_signals\percussive.wav)

  Raw results:
        signal  invalid  rms_median  rms_std  crest_median    crest_std  dynamic_range  energy_variance  energy_mod_rate  attack_time  attack_slope  decay_slope
          sine    False    0.353548 0.000360      1.414237 1.439192e-03       0.023772     1.999437e-03         0.000000     0.000000           0.0     0.000000
       am_sine    False    0.359312 0.188508      1.975132 3.351374e-01      15.104318     1.041631e+00         0.003636     0.185760           0.0     0.000000
   white_noise    False    0.480829 0.005832      2.079739 2.525359e-02       0.2624

In [10]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [11]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  AMPLITUDE/ENERGY FEATURE TESTBED REPORT
  Total: 36 | PASS: 36 | FAIL: 0 | SKIPPED: 0

  ✅ sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  rms_median                    0.353548       [0.3182, 0.3889]     ✅ PASS
  crest_median                  1.414237       [1.3000, 1.6000]     ✅ PASS
  dynamic_range                 0.023772       [0.0000, 3.0000]     ✅ PASS
  energy_variance               0.001999       [0.0000, 0.1500]     ✅ PASS
  energy_mod_rate               0.000000       [0.0000, 0.0100]     ✅ PASS

  ✅ am_sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  rms_median  

In [12]:
# Add this temporarily to extract_features():
import librosa
sig = AudioSignal("test_signals/sine.wav", N=N, H=H)
onset_env = librosa.onset.onset_strength(y=sig.y, sr=sig.sr, hop_length=H)
onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sig.sr, hop_length=H, units='frames')
print(f"  sine: onset_env max={onset_env.max():.4f}, num_onsets={len(onsets)}")

  sine: onset_env max=0.0486, num_onsets=1


In [13]:
# In extract_features(), for click_train only:
sig = AudioSignal("test_signals/click_train.wav", N=N, H=H)
tf = TimeFeatures(sig)
rms = tf._rms_envelope()
peak = tf._peak_amplitude()
mask = tf._active_rms_mask()

rms_db = 20.0 * np.log10(rms + 1e-10)
peak_db = 20.0 * np.log10(peak + 1e-10)

print(f"  Total frames:  {len(mask)}")
print(f"  Active frames: {mask.sum()}")
print(f"  RMS  dB range: [{rms_db.min():.1f}, {rms_db.max():.1f}]")
print(f"  Peak dB range: [{peak_db.min():.1f}, {peak_db.max():.1f}]")
print(f"  Frames where peak_db > -60: {(peak_db > -60).sum()}")
print(f"  Frames where rms_db  > -60: {(rms_db  > -60).sum()}")

  Total frames:  126
  Active frames: 21
  RMS  dB range: [-200.0, -25.7]
  Peak dB range: [-200.0, -0.9]
  Frames where peak_db > -60: 21
  Frames where rms_db  > -60: 21


In [14]:
SR = 22050
DURATION = 3.0
N = 2048
H = 512
OUT_DIR = "dataset/test_signals_noise"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [15]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_low_freq_sine(freq=100.0, amp=0.5, duration=DURATION, sr=SR):
    """
    Low frequency sine - low ZCR, high periodicity (voiced).
    Expected ZCR = 2*freq/sr = 2*100/22050 ≈ 0.009
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_high_freq_sine(freq=5000.0, amp=0.5, duration=DURATION, sr=SR):
    """
    High frequency sine - high ZCR, still periodic (voiced).
    Expected ZCR = 2*5000/22050 ≈ 0.45
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amp * np.sin(2 * np.pi * freq * t)

def gen_white_noise(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    White noise - high ZCR (~0.5), no periodicity (unvoiced).
    Expected ZCR ≈ 0.5 (crosses zero ~50% of samples)
    """
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

def gen_bandpass_noise(flow=300.0, fhigh=3400.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Bandpass filtered noise (telephone band) - speech-like spectrum.
    Expected: moderate ZCR (0.2-0.4), low periodicity
    """
    from scipy.signal import butter, filtfilt
    
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(int(sr * duration))
    
    # Butterworth bandpass
    nyq = sr / 2.0
    low = flow / nyq
    high = fhigh / nyq
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, noise)
    
    # Normalize
    filtered = amp * filtered / (np.max(np.abs(filtered)) + EPS)
    return filtered

def gen_pulse_train(freq=10.0, duty_cycle=0.1, amp=0.5, duration=DURATION, sr=SR):
    """Square wave with amplitude envelope to create transients"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    square = np.sign(np.sin(2 * np.pi * freq * t))
    
    # Add sharp envelope at each transition to create energy transients
    period_samples = int(sr / freq)
    envelope = np.ones_like(square)
    
    for i in range(0, len(square), period_samples // 2):  # Each half-period
        # Sharp attack
        attack_len = min(100, len(envelope) - i)
        envelope[i:i+attack_len] = np.linspace(0.1, 1.0, attack_len)
    
    return amp * square * envelope

def gen_voiced_speech_proxy(f0=150.0, formants=[800, 1200, 2500], amp=0.5, duration=DURATION, sr=SR):
    """
    Voiced speech proxy - sum of harmonics with formant envelope.
    Expected: low ZCR, very high periodicity (VUR ≈ 1.0)
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    # Fundamental + harmonics
    sig = np.zeros_like(t)
    for h in range(1, 11):  # 10 harmonics
        harmonic_freq = h * f0
        if harmonic_freq < sr / 2:
            sig += (1.0 / h) * np.sin(2 * np.pi * harmonic_freq * t)
    
    # Formant-like amplitude envelope (simple gaussian bumps)
    envelope = np.ones_like(t)
    for formant in formants:
        envelope += 2.0 * np.exp(-((t % 1.0) - 0.5)**2 / 0.1)
    
    sig = sig * envelope
    sig = amp * sig / (np.max(np.abs(sig)) + EPS)
    return sig

def gen_unvoiced_fricative_proxy(center_freq=4000.0, bandwidth=2000.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Unvoiced fricative proxy - highpass filtered noise.
    Expected: very high ZCR (0.4-0.5), no periodicity (VUR ≈ 0.0)
    """
    from scipy.signal import butter, filtfilt
    
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(int(sr * duration))
    
    # Highpass filter
    nyq = sr / 2.0
    cutoff = center_freq / nyq
    b, a = butter(4, cutoff, btype='high')
    filtered = filtfilt(b, a, noise)
    
    filtered = amp * filtered / (np.max(np.abs(filtered)) + EPS)
    return filtered

def gen_percussive_burst_train(rate=4.0, amp=0.7, duration=DURATION, sr=SR, seed=42):
    """
    Periodic noise bursts - clear transients.
    Expected transient rate = rate (e.g., 4 Hz)
    """
    sig = np.zeros(int(sr * duration))
    period_samples = int(sr / rate)
    burst_len = int(sr * 0.05)  # 50ms bursts
    
    rng = np.random.default_rng(seed)
    
    for i in range(0, len(sig), period_samples):
        burst = amp * rng.standard_normal(burst_len)
        # Exponential envelope
        env = np.exp(-np.linspace(0, 3, burst_len))
        burst = burst * env
        
        end = min(i + burst_len, len(sig))
        sig[i:end] = burst[:end-i]
    
    return sig

def gen_alternating_voiced_unvoiced(segment_duration=0.5, f0=200.0, amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Alternating voiced (sine) and unvoiced (noise) segments.
    Expected: moderate VUR (≈ 0.5), high ZCR variance
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    sig = np.zeros_like(t)
    
    segment_samples = int(segment_duration * sr)
    rng = np.random.default_rng(seed)
    
    voiced = True
    for i in range(0, len(sig), segment_samples):
        end = min(i + segment_samples, len(sig))
        
        if voiced:
            # Voiced segment (sine)
            seg_t = t[i:end] - t[i]
            sig[i:end] = amp * np.sin(2 * np.pi * f0 * seg_t)
        else:
            # Unvoiced segment (noise)
            sig[i:end] = amp * rng.standard_normal(end - i)
        
        voiced = not voiced
    
    return sig

In [16]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    
    ZCR formula: 2 * freq / sr (for sine waves)
    VUR: 0 = fully unvoiced, 1 = fully voiced
    ZCR Variance: normalized variability
    Transient Rate: onsets per second
    """
    gt = {}

    # --- Low Frequency Sine (100 Hz) ---
    # ZCR = 2*100/22050 ≈ 0.009
    gt["low_freq_sine"] = {
        "zcr_median":       (0.005, 0.015),
        "zcr_variance":     (0.0, 0.1),      # Very stable
        "voiced_ratio":     (0.9, 1.0),      # Highly periodic
        "unvoiced_ratio":   (0.0, 0.1),
        "transient_rate":   (0.0, 0.5),      # No transients
    }

    # --- High Frequency Sine (5000 Hz) ---
    # ZCR = 2*5000/22050 ≈ 0.453
    gt["high_freq_sine"] = {
        "zcr_median":       (0.40, 0.50),
        "zcr_variance":     (0.0, 0.1),      # Stable
        "voiced_ratio":     (0.8, 1.0),      # Still periodic despite high ZCR
        "unvoiced_ratio":   (0.0, 0.2),
        "transient_rate":   (0.0, 0.5),
    }

    # --- White Noise ---
    # ZCR ≈ 0.5 (theoretical), no periodicity
    gt["white_noise"] = {
        "zcr_median":       (0.45, 0.55),
        "zcr_variance":     (0.0, 0.15),     # Low variance (always noisy)
        "voiced_ratio":     (0.0, 0.1),      # Not periodic
        "unvoiced_ratio":   (0.9, 1.0),
        "transient_rate":   (0.0, 2.0),      # Few false detections
    }

    # --- Bandpass Noise (300-3400 Hz) ---
    gt["bandpass_noise"] = {
        "zcr_median":       (0.15, 0.35),
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.0, 0.2),      # Noisy, not periodic
        "unvoiced_ratio":   (0.8, 1.0),
        "transient_rate":   (0.0, 3.0),
    }

    # --- Pulse Train (10 Hz, 10% duty cycle) ---
    # Many zero crossings per period
    gt["pulse_train"] = {
        "zcr_median":       (0.0005, 0.002),    # Depends on duty cycle
        "zcr_variance":     (0.0, 0.3),
        "voiced_ratio":     (0.7, 1.0),      # Periodic but many crossings
        "unvoiced_ratio":   (0.0, 0.3),
        "transient_rate":   (6.0, 12.0),     # ~10 Hz transients
    }

    # --- Voiced Speech Proxy (150 Hz fundamental) ---
    gt["voiced_speech"] = {
        "zcr_median":       (0.01, 0.05),    # Low freq harmonics dominate
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.85, 1.0),     # Highly periodic
        "unvoiced_ratio":   (0.0, 0.15),
        "transient_rate":   (0.0, 1.0),
    }

    # --- Unvoiced Fricative Proxy ---
    gt["unvoiced_fricative"] = {
        "zcr_median":       (0.55, 0.75),    # High freq noise
        "zcr_variance":     (0.0, 0.2),
        "voiced_ratio":     (0.0, 0.1),      # No periodicity
        "unvoiced_ratio":   (0.9, 1.0),
        "transient_rate":   (0.0, 4.0),
    }

    # --- Percussive Burst Train (4 Hz) ---
    gt["percussive_bursts"] = {
        "zcr_median":       (0.03, 0.15),    # Noise bursts
        "zcr_variance":     (0.1, 1.0),      # High variance (bursts vs silence)
        "voiced_ratio":     (0.0, 0.2),      # Not periodic
        "unvoiced_ratio":   (0.8, 1.0),
        "transient_rate":   (3.0, 5.0),      # ~4 Hz
    }

    # --- Alternating Voiced/Unvoiced ---
    gt["alternating"] = {
        "zcr_median":       (0.10, 0.35),    # Mix of both
        "zcr_variance":     (0.3, 2.0),      # High variance (alternates)
        "voiced_ratio":     (0.4, 0.7),      # Mix
        "unvoiced_ratio":   (0.3, 0.6),
        "transient_rate":   (0.0, 4.0),      # Transitions
    }

    return gt

In [17]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals as .wav files."""
    signals = {
        "low_freq_sine":        gen_low_freq_sine(),
        "high_freq_sine":       gen_high_freq_sine(),
        "white_noise":          gen_white_noise(),
        "bandpass_noise":       gen_bandpass_noise(),
        "pulse_train":          gen_pulse_train(),
        "voiced_speech":        gen_voiced_speech_proxy(),
        "unvoiced_fricative":   gen_unvoiced_fricative_proxy(),
        "percussive_bursts":    gen_percussive_burst_train(),
        "alternating":          gen_alternating_voiced_unvoiced(),
    }
    
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    
    return paths

In [18]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Extract noise/speech features from each test signal.
    
    ADAPT: uncomment import and point to your module.
    """
    # from your_module import AudioSignal, TimeFeatures

    results = []

    for name, path in file_paths.items():
        print(f"  Processing: {name}")

        try:
            sig = AudioSignal(path, N=N, H=H)

            if sig.invalid:
                print(f"    [SKIP] Invalid signal: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    'zcr_median': np.nan,
                    'zcr_variance': np.nan,
                    'voiced_ratio': np.nan,
                    'unvoiced_ratio': np.nan,
                    'transient_rate': np.nan,
                    'transient_count': np.nan,
                })
                continue

            tf = TimeFeatures(sig)

            # Extract features
            zcr = tf._zero_crossing_rate()
            mask = tf._active_rms_mask()

            results.append({
                'signal':           name,
                'invalid':          False,
                'zcr_median':       float(np.median(zcr)),
                'zcr_variance':     float(tf._zcr_variance()),
                'voiced_ratio':     float(tf._voiced_ratio()),
                'unvoiced_ratio':   float(tf._unvoiced_ratio()),
                'transient_rate':   float(tf._transient_rate()),
                'transient_count':  int(tf._transient_counts()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                'zcr_median': np.nan,
                'zcr_variance': np.nan,
                'voiced_ratio': np.nan,
                'unvoiced_ratio': np.nan,
                'transient_rate': np.nan,
                'transient_count': np.nan,
            })

    return pd.DataFrame(results)

In [19]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  NOISE/SPEECH INDICATORS TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.4f}, {r['expected_max']:.4f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [20]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved dataset/test_signals_noise\low_freq_sine.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\high_freq_sine.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\white_noise.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\bandpass_noise.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\pulse_train.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\voiced_speech.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\unvoiced_fricative.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\percussive_bursts.wav | 66150 samples | 3.00s
  Saved dataset/test_signals_noise\alternating.wav | 66150 samples | 3.00s


In [21]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: low_freq_sine
  Processing: high_freq_sine
  Processing: white_noise
  Processing: bandpass_noise
  Processing: pulse_train
  Processing: voiced_speech
  Processing: unvoiced_fricative
  Processing: percussive_bursts
  Processing: alternating

  Raw results:
            signal  invalid  zcr_median  zcr_variance  voiced_ratio  unvoiced_ratio  transient_rate  transient_count
     low_freq_sine    False    0.009277      0.052632      1.000000    7.936984e-13        0.000000                0
    high_freq_sine    False    0.453125      0.001078      1.000000    7.936984e-13        0.000000                0
       white_noise    False    0.497314      0.033382      0.000000    1.000000e+00        1.333333                4
    bandpass_noise    False    0.176758      0.037293      0.000000    1.000000e+00        1.333333                4
       pulse_train    False    0.000977      0.000000      1.000000    7.936984e-13        9.333333             

In [22]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [23]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  NOISE/SPEECH INDICATORS TESTBED REPORT
  Total: 45 | PASS: 45 | FAIL: 0 | SKIPPED: 0

  ✅ low_freq_sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  zcr_median                    0.009277       [0.0050, 0.0150]     ✅ PASS
  zcr_variance                  0.052632       [0.0000, 0.1000]     ✅ PASS
  voiced_ratio                  1.000000       [0.9000, 1.0000]     ✅ PASS
  unvoiced_ratio                0.000000       [0.0000, 0.1000]     ✅ PASS
  transient_rate                0.000000       [0.0000, 0.5000]     ✅ PASS

  ✅ high_freq_sine (5/5 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────

In [24]:
# Add to testbed temporarily:
sig = AudioSignal("dataset/test_signals_noise/pulse_train.wav", N=N, H=H)
print(f"pulse_train: min={sig.y.min():.4f} max={sig.y.max():.4f} mean={sig.y.mean():.4f}")
print(f"  unique values: {np.unique(sig.y)}")

pulse_train: min=-0.5000 max=0.5000 mean=-0.0001
  unique values: [-0.5        -0.4954834  -0.49093628 -0.48638916 -0.48449707 -0.48184204
 -0.47729492 -0.4727478  -0.46899414 -0.46820068 -0.46365356 -0.45910645
 -0.45455933 -0.4534607  -0.4500122  -0.4454651  -0.44091797 -0.43795776
 -0.43637085 -0.43182373 -0.4272766  -0.4227295  -0.42242432 -0.41818237
 -0.41366577 -0.40911865 -0.4069214  -0.40457153 -0.4000244  -0.3954773
 -0.39138794 -0.39093018 -0.38638306 -0.38183594 -0.37728882 -0.375885
 -0.3727417  -0.36819458 -0.36364746 -0.36035156 -0.35910034 -0.35455322
 -0.3500061  -0.34545898 -0.34484863 -0.34091187 -0.33636475 -0.33184814
 -0.3293152  -0.32730103 -0.3227539  -0.3182068  -0.31381226 -0.31365967
 -0.30911255 -0.30456543 -0.3000183  -0.2982788  -0.2954712  -0.29092407
 -0.28637695 -0.28277588 -0.28182983 -0.2772827  -0.2727356  -0.26818848
 -0.26724243 -0.26364136 -0.25909424 -0.25454712 -0.2517395  -0.25
 -0.2454834  -0.24093628 -0.23638916 -0.23623657 -0.23184204 -0.227

In [25]:
SR = 22050
DURATION = 10.0
N = 2048
H = 512
OUT_DIR = "dataset/test_signals_rhythm"
EPS = 1e-10

os.makedirs(OUT_DIR, exist_ok=True)

In [26]:
# =============================================================================
# SIGNAL GENERATORS
# =============================================================================

def gen_metronomic_clicks(bpm=120.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Perfect metronomic click train.
    
    Known properties:
    - Tempo = bpm (exact)
    - IOI = 60/bpm seconds (constant)
    - CV(IOI) = 0 (perfect regularity)
    - Pulse clarity = very high (>0.8)
    - Stability = 1.0 (no tempo variation)
    """
    sig = np.zeros(int(sr * duration))
    period_sec = 60.0 / bpm
    period_samples = int(sr * period_sec)
    
    # Place clicks
    for i in range(0, len(sig), period_samples):
        # Short impulse with exponential decay
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] = click
    
    return sig

def gen_accelerando(bpm_start=80.0, bpm_end=160.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Gradual tempo increase (accelerando).
    
    Known properties:
    - Starting tempo ≈ bpm_start
    - Ending tempo ≈ bpm_end
    - IOI decreases linearly
    - CV(IOI) > 0.3 (high variance)
    - Stability < 0.5 (low - tempo changes)
    """
    sig = np.zeros(int(sr * duration))
    
    t = 0.0  # Current time in seconds
    current_bpm = bpm_start
    bpm_rate = (bpm_end - bpm_start) / duration  # BPM change per second
    
    while t < duration:
        # Place click
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Calculate next IOI based on current tempo
        ioi = 60.0 / current_bpm
        t += ioi
        
        # Update tempo
        current_bpm += bpm_rate * ioi
    
    return sig

def gen_irregular_rhythm(avg_ioi=0.5, ioi_std=0.2, amp=0.7, duration=DURATION, sr=SR, seed=42):
    """
    Random IOIs with Gaussian distribution (rubato / free time).
    
    Known properties:
    - Mean IOI ≈ avg_ioi
    - Std(IOI) ≈ ioi_std
    - CV(IOI) = ioi_std / avg_ioi
    - Pulse clarity < 0.3 (weak/no pulse)
    - Stability < 0.5 (irregular)
    """
    rng = np.random.default_rng(seed)
    sig = np.zeros(int(sr * duration))
    
    t = 0.0
    while t < duration:
        # Place click
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Random IOI (clipped to positive)
        ioi = max(0.1, rng.normal(avg_ioi, ioi_std))
        t += ioi
    
    return sig

def gen_polyrhythm_3_over_2(bpm_base=120.0, amp=0.7, duration=DURATION, sr=SR):
    """
    3:2 polyrhythm (3 beats against 2 beats in same time span).
    
    Known properties:
    - Two IOI modes: 60/(bpm_base) and 60/(bpm_base*1.5)
    - Pulse clarity medium (0.4-0.7) - multiple periodicities
    - Beat periodicity < 0.7 (bimodal IOI distribution)
    """
    sig = np.zeros(int(sr * duration))
    
    # Stream 1: Base tempo
    period_1 = int(sr * 60.0 / bpm_base)
    for i in range(0, len(sig), period_1):
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] += click * 0.7  # Slightly quieter
    
    # Stream 2: 1.5x faster
    period_2 = int(sr * 60.0 / (bpm_base * 1.5))
    for i in range(0, len(sig), period_2):
        click_len = min(100, len(sig) - i)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] += click * 0.5  # Even quieter
    
    # Normalize
    sig = np.clip(sig, -1.0, 1.0)
    return sig

def gen_syncopated_pattern(bpm=120.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Syncopated rhythm (strong offbeats, weak downbeats).
    Pattern: weak-STRONG-weak-STRONG per measure
    
    Known properties:
    - Tempo = bpm
    - IOI = constant (60/bpm)
    - Pulse clarity medium-high (0.5-0.8) - clear pulse but accents vary
    - Stability high (>0.8) - tempo is stable
    """
    sig = np.zeros(int(sr * duration))
    period_sec = 60.0 / bpm
    period_samples = int(sr * period_sec)
    
    # Pattern: weak, strong, weak, strong (4 beats per measure)
    pattern = [0.3, 0.9, 0.4, 0.9]
    
    beat_idx = 0
    for i in range(0, len(sig), period_samples):
        # Accent based on pattern
        accent = pattern[beat_idx % len(pattern)]
        
        click_len = min(100, len(sig) - i)
        click = amp * accent * np.exp(-np.linspace(0, 5, click_len))
        sig[i:i+click_len] = click
        
        beat_idx += 1
    
    return sig

def gen_swing_rhythm(bpm=120.0, swing_ratio=2.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Swing/shuffle rhythm (long-short pattern).
    
    Known properties:
    - Average tempo ≈ bpm
    - IOI alternates: long (swing_ratio × base) and short
    - CV(IOI) moderate (0.2-0.4)
    - Pulse clarity high (0.7-0.9) - strong periodic pattern
    """
    sig = np.zeros(int(sr * duration))
    
    # Base eighth note duration
    base_ioi = 60.0 / (bpm * 2)  # bpm is quarter notes, we want eighths
    
    # Swing: first note is longer, second is shorter
    # Ratio of 2:1 means triplet feel (2/3 + 1/3 of beat)
    total_time = base_ioi * 2
    ioi_long = total_time * (swing_ratio / (swing_ratio + 1))
    ioi_short = total_time * (1 / (swing_ratio + 1))
    
    t = 0.0
    is_long = True
    
    while t < duration:
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        # Alternate long/short
        t += ioi_long if is_long else ioi_short
        is_long = not is_long
    
    return sig

def gen_ritardando(bpm_start=140.0, bpm_end=70.0, amp=0.7, duration=DURATION, sr=SR):
    """
    Gradual slowdown (ritardando).
    
    Known properties:
    - Starting tempo ≈ bpm_start
    - Ending tempo ≈ bpm_end
    - IOI increases over time
    - Stability < 0.5
    """
    sig = np.zeros(int(sr * duration))
    
    t = 0.0
    current_bpm = bpm_start
    bpm_rate = (bpm_end - bpm_start) / duration
    
    while t < duration:
        sample_idx = int(t * sr)
        if sample_idx >= len(sig):
            break
        
        click_len = min(100, len(sig) - sample_idx)
        click = amp * np.exp(-np.linspace(0, 5, click_len))
        sig[sample_idx:sample_idx+click_len] = click
        
        ioi = 60.0 / current_bpm
        t += ioi
        current_bpm += bpm_rate * ioi
    
    return sig

def gen_no_rhythm(amp=0.5, duration=DURATION, sr=SR, seed=42):
    """
    Continuous noise with no rhythmic structure.
    
    Known properties:
    - No clear tempo
    - Few/no onsets (depends on energy threshold)
    - Pulse clarity ≈ 0
    - All rhythm metrics should be near-zero or undefined
    """
    rng = np.random.default_rng(seed)
    return amp * rng.standard_normal(int(sr * duration))

In [ ]:
# =============================================================================
# GROUND TRUTH
# =============================================================================

def compute_ground_truth():
    """
    Expected feature ranges per signal.
    
    Key metrics:
    - onset_rate: onsets/second
    - tempo: BPM from autocorrelation
    - ioi_mean: average inter-onset-interval (seconds)
    - ioi_cv: coefficient of variation (std/mean)
    - pulse_clarity: 0-1 (AC peak dominance)
    - stability_exp: 0-1 (tempo consistency)
    - periodicity: 0-1 (IOI entropy-based)
    """
    gt = {}

    # --- Metronomic Clicks (120 BPM) ---
    # IOI = 60/120 = 0.5 sec
    # Onset rate = 120/60 = 2 Hz
    gt["metronomic_120"] = {
        "onset_rate":       (1.8, 2.2),      # ~2 Hz
        "tempo":            (115.0, 125.0),  # 120 BPM
        "ioi_mean":         (0.48, 0.52),    # 0.5 sec
        "ioi_cv":           (0.0, 0.05),     # Nearly zero variance
        "pulse_clarity":    (0.75, 1.0),     # Very high
        "stability_exp":    (0.95, 1.0),     # Perfect stability
        "periodicity":      (0.7, 1.0),     # Highly periodic
    }

    # --- Accelerando (80 → 160 BPM) ---
    # Average tempo ≈ 120, but high variance
    gt["accelerando"] = {
        "onset_rate":       (1.5, 3.0),      # Increases over time
        "tempo":            (140.0, 180.0),  # Global estimate varies
        "ioi_mean":         (0.4, 0.7),      # Average IOI
        "ioi_cv":           (0.15, 0.50),     # High variance
        "pulse_clarity":    (0.3, 0.7),      # Moderate (tempo change hurts AC)
        "stability_exp":    (0.0, 0.4),      # Low (tempo changes)
        "periodicity":      (0.15, 0.8),      # Moderate
    }

    # --- Irregular Rhythm (avg_ioi=0.5, std=0.2) ---
    # CV = 0.2/0.5 = 0.4
    gt["irregular"] = {
        "onset_rate":       (1.5, 2.5),      # ~2 Hz average
        "tempo":            (80.0, 160.0),   # Weak/unreliable
        "ioi_mean":         (0.4, 0.6),      # 0.5 target
        "ioi_cv":           (0.3, 0.5),      # 0.4 target
        "pulse_clarity":    (0.0, 0.3),      # Very weak
        "stability_exp":    (0.0, 0.3),      # Low
        "periodicity":      (0.0, 0.4),      # Low
    }

    # --- Polyrhythm 3:2 (base 120 BPM) ---
    # Two IOI modes: 0.5 sec (120 BPM) and 0.333 sec (180 BPM)
    gt["polyrhythm"] = {
        "onset_rate":       (3.0, 5.5),      # More onsets (two streams)
        "tempo":            (100.0, 140.0),  # May lock to either stream
        "ioi_mean":         (0.20, 0.30),     # Mix of both periods
        "ioi_cv":           (0.2, 0.5),      # Bimodal distribution
        "pulse_clarity":    (0.3, 0.7),      # Moderate (competing pulses)
        "stability_exp":    (0.6, 0.95),     # Stable (both streams regular)
        "periodicity":      (0.3, 0.7),      # Lower (bimodal IOI)
    }

    # --- Syncopated (120 BPM) ---
    # Same IOI as metronomic, just different accents
    gt["syncopated"] = {
        "onset_rate":       (1.8, 2.2),      # 2 Hz
        "tempo":            (115.0, 125.0),  # 120 BPM
        "ioi_mean":         (0.48, 0.52),    # 0.5 sec
        "ioi_cv":           (0.0, 0.05),     # Regular timing
        "pulse_clarity":    (0.5, 0.85),     # High (accents don't destroy pulse)
        "stability_exp":    (0.9, 1.0),      # Very stable
        "periodicity":      (0.7, 1.0),      # Highly periodic
    }

    # --- Swing Rhythm (120 BPM, 2:1 ratio) ---
    # IOIs alternate ~0.4 and ~0.2 sec
    gt["swing"] = {
        "onset_rate":       (3.5, 4.5),      # ~4 Hz (eighth notes)
        "tempo":            (115.0, 125.0),  # 120 BPM (quarter note)
        "ioi_mean":         (0.22, 0.28),    # Average of long/short
        "ioi_cv":           (0.25, 0.45),    # Moderate (alternating pattern)
        "pulse_clarity":    (0.65, 0.95),    # High (strong periodic pattern)
        "stability_exp":    (0.85, 1.0),     # Stable
        "periodicity":      (0.55, 0.9),      # High but bimodal IOI
    }

    # --- Ritardando (140 → 70 BPM) ---
    gt["ritardando"] = {
        "onset_rate":       (1.2, 2.5),      # Decreases over time
        "tempo":            (110.0, 140.0),   # Global estimate
        "ioi_mean":         (0.5, 0.9),      # Increases
        "ioi_cv":           (0.15, 0.6),      # High variance
        "pulse_clarity":    (0.3, 0.7),      # Moderate
        "stability_exp":    (0.0, 0.4),      # Low
        "periodicity":      (0.15, 0.8),      # Moderate
    }

    # --- No Rhythm (noise) ---
    gt["no_rhythm"] = {
        "onset_rate":       (0.0, 5.0),      # Few/no onsets
        "tempo":            (0.0, 240.0),    # Meaningless
        "ioi_mean":         (0.0, 5.0),      # Undefined/unreliable
        "ioi_cv":           (0.0, 2.0),      # Undefined
        "pulse_clarity":    (0.0, 0.2),      # No pulse
        "stability_exp":    (0.0, 0.5),      # Undefined
        "periodicity":      (0.0, 0.3),      # No structure
    }

    return gt

In [28]:
# =============================================================================
# GENERATE + SAVE
# =============================================================================

def generate_all_signals():
    """Generate and save all test signals."""
    signals = {
        "metronomic_120":   gen_metronomic_clicks(bpm=120.0),
        "accelerando":      gen_accelerando(bpm_start=80.0, bpm_end=160.0),
        "irregular":        gen_irregular_rhythm(avg_ioi=0.5, ioi_std=0.2),
        "polyrhythm":       gen_polyrhythm_3_over_2(bpm_base=120.0),
        "syncopated":       gen_syncopated_pattern(bpm=120.0),
        "swing":            gen_swing_rhythm(bpm=120.0, swing_ratio=2.0),
        "ritardando":       gen_ritardando(bpm_start=140.0, bpm_end=70.0),
        "no_rhythm":        gen_no_rhythm(),
    }
    
    paths = {}
    for name, sig in signals.items():
        path = os.path.join(OUT_DIR, f"{name}.wav")
        sf.write(path, sig.astype(np.float32), SR)
        print(f"  Saved {path} | {len(sig)} samples | {len(sig)/SR:.2f}s")
        paths[name] = path
    
    return paths

In [29]:
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def extract_features(file_paths):
    """
    Extract rhythm/beat features from each test signal.
    
    ADAPT: uncomment import and point to your module.
    """
    # from your_module import AudioSignal, TimeFeatures

    results = []

    for name, path in file_paths.items():
        print(f"  Processing: {name}")

        try:
            sig = AudioSignal(path, N=N, H=H)

            if sig.invalid:
                print(f"    [SKIP] Invalid signal: {name}")
                results.append({
                    'signal': name,
                    'invalid': True,
                    **{k: np.nan for k in ['onset_rate', 'tempo', 'ioi_mean', 
                                            'ioi_cv', 'pulse_clarity', 
                                            'stability_exp', 'periodicity']}
                })
                continue

            tf = TimeFeatures(sig)

            # Add to extract_features() for debugging:
            if name in ["metronomic_120", "irregular", "accelerando", "no_rhythm"]:
                ac = tf._onset_autocorrelation()
                print(f"\n  {name} AC diagnostic:")
                print(f"    AC[0]={ac[0]:.6f}")
                print(f"    AC[1:11]={ac[1:11]}")
                print(f"    AC max(1:)={np.max(ac[1:]):.6f}, mean(1:)={np.mean(ac[1:]):.6f}")
                
                tempos = tf._windowed_tempo_series()
                print(f"    Windowed tempos: {tempos}")
                print(f"    Tempo std: {np.std(tempos):.4f}")

            # Extract features
            ioi_mean, ioi_std, ioi_cv = tf._ioi_stats()
            stability_dict = tf._rhythmic_stability()

            results.append({
                'signal':           name,
                'invalid':          False,
                'onset_rate':       float(tf._onset_rate()),
                'tempo':            float(tf._tempo_from_onset_ac()),
                'ioi_mean':         float(ioi_mean),
                'ioi_cv':           float(ioi_cv),
                'pulse_clarity':    float(tf._pulse_clarity_ac()),
                'stability_exp':    float(stability_dict['stability_exp']),
                'periodicity':      float(tf._beat_periodicity_entropy()),
            })

        except Exception as e:
            print(f"    [ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'signal': name,
                'invalid': True,
                **{k: np.nan for k in ['onset_rate', 'tempo', 'ioi_mean', 
                                        'ioi_cv', 'pulse_clarity', 
                                        'stability_exp', 'periodicity']}
            })

    return pd.DataFrame(results)

In [30]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate(df, ground_truth):
    """Compare extracted features against ground truth ranges."""
    rows = []
    for _, row in df.iterrows():
        name = row['signal']
        if name not in ground_truth:
            continue
        
        gt = ground_truth[name]
        for feature, (expected_min, expected_max) in gt.items():
            if feature not in row or pd.isna(row[feature]):
                rows.append({
                    'signal': name,
                    'feature': feature,
                    'actual': None,
                    'expected_min': expected_min,
                    'expected_max': expected_max,
                    'status': 'SKIP (invalid)'
                })
                continue

            actual = row[feature]
            passed = expected_min <= actual <= expected_max
            rows.append({
                'signal': name,
                'feature': feature,
                'actual': round(actual, 6),
                'expected_min': expected_min,
                'expected_max': expected_max,
                'status': 'PASS' if passed else 'FAIL'
            })

    return pd.DataFrame(rows)


def print_report(validation_df):
    """Pretty-print the validation report."""
    total = len(validation_df)
    passed = (validation_df['status'] == 'PASS').sum()
    failed = (validation_df['status'] == 'FAIL').sum()
    skipped = validation_df['status'].str.contains('SKIP').sum()

    print("\n" + "=" * 80)
    print("  RHYTHM/BEAT FEATURE TESTBED REPORT")
    print("=" * 80)
    print(f"  Total: {total} | PASS: {passed} | FAIL: {failed} | SKIPPED: {skipped}")
    print("=" * 80)

    for signal_name in validation_df['signal'].unique():
        subset = validation_df[validation_df['signal'] == signal_name]
        sig_pass = (subset['status'] == 'PASS').sum()
        sig_total = len(subset)
        icon = "✅" if sig_pass == sig_total else "❌"

        print(f"\n  {icon} {signal_name} ({sig_pass}/{sig_total} passed)")
        print(f"  {'─' * 74}")
        print(f"  {'Feature':<25} {'Actual':>12} {'Expected Range':>22} {'Status':>10}")
        print(f"  {'─' * 74}")

        for _, r in subset.iterrows():
            if r['status'] == 'PASS':
                s_icon = "✅ PASS"
            elif 'SKIP' in str(r['status']):
                s_icon = "⚠️  SKIP"
            else:
                s_icon = "❌ FAIL"

            actual_str = f"{r['actual']:.6f}" if r['actual'] is not None else "N/A"
            range_str = f"[{r['expected_min']:.2f}, {r['expected_max']:.2f}]"
            print(f"  {r['feature']:<25} {actual_str:>12} {range_str:>22} {s_icon:>10}")

    print("\n" + "=" * 80)

In [31]:
print("\n[1/4] Generating test signals...")
file_paths = generate_all_signals()


[1/4] Generating test signals...
  Saved dataset/test_signals_rhythm\metronomic_120.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\accelerando.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\irregular.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\polyrhythm.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\syncopated.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\swing.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\ritardando.wav | 220500 samples | 10.00s
  Saved dataset/test_signals_rhythm\no_rhythm.wav | 220500 samples | 10.00s


In [32]:
print("\n[2/4] Extracting features...")
df = extract_features(file_paths)
print("\n  Raw results:")
print(df.to_string(index=False))


[2/4] Extracting features...
  Processing: metronomic_120

  metronomic_120 AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.21377888 -0.06142894 -0.07434491 -0.07451621 -0.07468751 -0.07485881
 -0.07503012 -0.07520141 -0.07537272 -0.07554402]
    AC max(1:)=0.876985, mean(1:)=-0.001163
    Windowed tempos: [117.45383523 117.45383523 117.45383523]
    Tempo std: 0.0000
  Processing: accelerando

  accelerando AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.16728118 -0.06329283 -0.06883483 -0.06899343 -0.06915203 -0.06931064
 -0.06946925 -0.06962785 -0.06937096 -0.0663236 ]
    AC max(1:)=0.167281, mean(1:)=-0.001163
    Windowed tempos: [129.19921875 151.99908088 151.99908088]
    Tempo std: 10.7480
  Processing: irregular

  irregular AC diagnostic:
    AC[0]=1.000000
    AC[1:11]=[ 0.25251568 -0.06306981 -0.08396249 -0.06774172 -0.01499858 -0.07161193
 -0.08053927 -0.08073273 -0.07241797 -0.01740324]
    AC max(1:)=0.252516, mean(1:)=-0.001163
    Windowed tempos: [129.19921875

In [33]:
print("\n[3/4] Validating against ground truth...")
gt = compute_ground_truth()
validation = validate(df, gt)


[3/4] Validating against ground truth...


In [34]:
print("\n[4/4] Report:")
print_report(validation)


[4/4] Report:

  RHYTHM/BEAT FEATURE TESTBED REPORT
  Total: 56 | PASS: 37 | FAIL: 19 | SKIPPED: 0

  ❌ metronomic_120 (6/7 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feature                         Actual         Expected Range     Status
  ──────────────────────────────────────────────────────────────────────────
  onset_rate                    1.900000           [1.80, 2.20]     ✅ PASS
  tempo                       117.453835       [115.00, 125.00]     ✅ PASS
  ioi_mean                      0.500519           [0.48, 0.52]     ✅ PASS
  ioi_cv                        0.023721           [0.00, 0.05]     ✅ PASS
  pulse_clarity                 0.958296           [0.75, 1.00]     ✅ PASS
  stability_exp                 1.000000           [0.95, 1.00]     ✅ PASS
  periodicity                   0.770687           [0.85, 1.00]     ❌ FAIL

  ❌ accelerando (2/7 passed)
  ──────────────────────────────────────────────────────────────────────────
  Feat